In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, regexp_replace
from pyspark.ml.feature import StringIndexer, VectorAssembler, MinMaxScaler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName("SocialMediaDataPrep").getOrCreate()

In [4]:
# 1 Leia o arquivo 'videos-tratados.snappy.parquet' no dataframe 'df_video'
df_video = spark.read.parquet('videos-comments-tratados.snappy.parquet')

df_video = df_video.dropDuplicates(["Video ID"])

df_video = df_video.drop("Comment", "Sentiment", "Likes Comment")

df_video = df_video.cache()
df_video.show()

+-----------+--------------------+------------+----------------+------+--------+--------+-----------+----+
|   Video ID|               Title|Published At|         Keyword| Likes|Comments|   Views|Interaction|Year|
+-----------+--------------------+------------+----------------+------+--------+--------+-----------+----+
|115amzVdV44|How To Fix a Wate...|  2020-08-18|          how-to|910553|   81975|52061447|   53053975|2020|
|2WPA1L9uJqo|What is the Schrö...|  2022-04-14|         physics| 11420|     896|  596902|     609218|2022|
|Rk4bAofG8xE|Pragg Beats Super...|  2022-08-18|           chess|  8534|     517|  254621|     263672|2022|
|U3DNz5asasA|Pixel Watch, Pixe...|  2022-05-11|          google|102993|    6346| 2568894|    2678233|2022|
|UI9I2p71ct0|Magical Realism I...|  2021-05-28|      literature|  2459|     162|   72310|      74931|2021|
|V6hofBnlJLY|Friday Finance | ...|  2022-08-18|         finance|    16|       3|   83121|      83140|2022|
|XWv_4L1_Z7Q|The Evolution of ...|  2

In [5]:
# 2 Adicione a coluna 'Month' com o valor do mês da coluna "Published At"
df_video = df_video.withColumn("Month", month(col("Published At")))
df_video.show()

+-----------+--------------------+------------+----------------+------+--------+--------+-----------+----+-----+
|   Video ID|               Title|Published At|         Keyword| Likes|Comments|   Views|Interaction|Year|Month|
+-----------+--------------------+------------+----------------+------+--------+--------+-----------+----+-----+
|115amzVdV44|How To Fix a Wate...|  2020-08-18|          how-to|910553|   81975|52061447|   53053975|2020|    8|
|2WPA1L9uJqo|What is the Schrö...|  2022-04-14|         physics| 11420|     896|  596902|     609218|2022|    4|
|Rk4bAofG8xE|Pragg Beats Super...|  2022-08-18|           chess|  8534|     517|  254621|     263672|2022|    8|
|U3DNz5asasA|Pixel Watch, Pixe...|  2022-05-11|          google|102993|    6346| 2568894|    2678233|2022|    5|
|UI9I2p71ct0|Magical Realism I...|  2021-05-28|      literature|  2459|     162|   72310|      74931|2021|    5|
|V6hofBnlJLY|Friday Finance | ...|  2022-08-18|         finance|    16|       3|   83121|      8

In [6]:
# 3 Adicione a coluna "Keyword Index" com a transformação da coluna 'keyword' para valores numéricos
indexer = StringIndexer(inputCol="keyword", outputCol="Keyword Index")
df_video = indexer.fit(df_video).transform(df_video)
df_video.show()

+-----------+--------------------+------------+----------------+------+--------+--------+-----------+----+-----+-------------+
|   Video ID|               Title|Published At|         Keyword| Likes|Comments|   Views|Interaction|Year|Month|Keyword Index|
+-----------+--------------------+------------+----------------+------+--------+--------+-----------+----+-----+-------------+
|115amzVdV44|How To Fix a Wate...|  2020-08-18|          how-to|910553|   81975|52061447|   53053975|2020|    8|         19.0|
|2WPA1L9uJqo|What is the Schrö...|  2022-04-14|         physics| 11420|     896|  596902|     609218|2022|    4|          7.0|
|Rk4bAofG8xE|Pragg Beats Super...|  2022-08-18|           chess|  8534|     517|  254621|     263672|2022|    8|         26.0|
|U3DNz5asasA|Pixel Watch, Pixe...|  2022-05-11|          google|102993|    6346| 2568894|    2678233|2022|    5|         29.0|
|UI9I2p71ct0|Magical Realism I...|  2021-05-28|      literature|  2459|     162|   72310|      74931|2021|    5

In [7]:
# Convertendo a coluna 'Year' para tipo inteiro para não ter erro
df_video = df_video.withColumn("Year", col("Year").cast("integer"))

# 4 Criando o vetor "Features" com os campos numéricos
assembler = VectorAssembler(
    inputCols=["Likes", "Views", "Year", "Month", "Keyword Index"],
    outputCol="Features"
)

df_video = assembler.transform(df_video)

In [8]:
# 5 Adicione a coluna "Features Normal" com os dados normalizados (removendo nulos)
df_video = df_video.dropna(subset=["Features", "Comments"])

scaler = MinMaxScaler(inputCol="Features", outputCol="Features Normal")
scaler_model = scaler.fit(df_video)
df_video = scaler_model.transform(df_video)
df_video.show()

+-----------+--------------------+------------+----------------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+
|   Video ID|               Title|Published At|         Keyword| Likes|Comments|   Views|Interaction|Year|Month|Keyword Index|            Features|     Features Normal|
+-----------+--------------------+------------+----------------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+
|115amzVdV44|How To Fix a Wate...|  2020-08-18|          how-to|910553|   81975|52061447|   53053975|2020|    8|         19.0|[910553.0,5.20614...|[0.05536777436388...|
|2WPA1L9uJqo|What is the Schrö...|  2022-04-14|         physics| 11420|     896|  596902|     609218|2022|    4|          7.0|[11420.0,596902.0...|[6.94473200941360...|
|Rk4bAofG8xE|Pragg Beats Super...|  2022-08-18|           chess|  8534|     517|  254621|     263672|2022|    8|         26.0|[8534.0,254621.0,...|[5.18985

In [9]:
# 6 Adicione a coluna "Features PCA" com a redução de 5 características para 1
pca = PCA(k=1, inputCol="Features Normal", outputCol="Features PCA")
pca_model = pca.fit(df_video)
df_video = pca_model.transform(df_video)
df_video.show()

+-----------+--------------------+------------+----------------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|   Video ID|               Title|Published At|         Keyword| Likes|Comments|   Views|Interaction|Year|Month|Keyword Index|            Features|     Features Normal|        Features PCA|
+-----------+--------------------+------------+----------------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|115amzVdV44|How To Fix a Wate...|  2020-08-18|          how-to|910553|   81975|52061447|   53053975|2020|    8|         19.0|[910553.0,5.20614...|[0.05536777436388...|[0.45760222504055...|
|2WPA1L9uJqo|What is the Schrö...|  2022-04-14|         physics| 11420|     896|  596902|     609218|2022|    4|          7.0|[11420.0,596902.0...|[6.94473200941360...|[0.11699006918633...|
|Rk4bAofG8xE|Pragg Beats Super...|  2022-08-18|   

In [10]:
# 7 Separe o dataframe df_video em 2 conjuntos: 80% para treinamento e 20% para teste
train_data, test_data = df_video.randomSplit([0.8, 0.2], seed=42)
# Visualiza  tamanho dos conjuntos
print("Tamanho do conjunto de treinamento:", train_data.count())
print("Tamanho do conjunto de teste:", test_data.count())

Tamanho do conjunto de treinamento: 1497
Tamanho do conjunto de teste: 372


In [11]:
# 8 Crie um modelo de regressão linear para estimar "Comments" e avalie o modelo
lr = LinearRegression(featuresCol="Features Normal", labelCol="Comments")
lr_model = lr.fit(train_data)

# Previsões no conjunto de teste
predictions = lr_model.transform(test_data)

# Avaliação com métricas de erro (RMSE) e ajuste (R²)
evaluator_rmse = RegressionEvaluator(labelCol="Comments", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(labelCol="Comments", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE(Erro de Quadrático médio da raiz): {rmse:.4f}")
print(f"R²(Coeficiente de Determinação): {r2:.4f}")

RMSE(Erro de Quadrático médio da raiz): 13146.3335
R²(Coeficiente de Determinação): 0.8056


In [12]:
# 9 Salve o dataframe df_video como 'videos-preparados-parquet' no formato parquet
df_video.write.mode("overwrite").parquet("videos-preparados-parquet")